# 01. EDA - MovieLens 100K

개인화 상품 추천 엔진 프로젝트의 Day 1 산출물입니다.  
이 노트북에서는 데이터 구조를 파악하고, 추천 실험에 필요한 평가 프레임워크를 준비합니다.

## 체크리스트
- [ ] MovieLens 100K 로딩
- [ ] 평점 분포 확인
- [ ] 유저/아이템 활동량 분포 확인
- [ ] 희소성 계산
- [ ] 시간 트렌드 확인
- [ ] Explicit / Implicit 피드백 데이터 구성
- [ ] Cold-start 기준 확인
- [ ] Train/Test 분할 전략 정리

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import DEFAULT_TOP_K, HIGH_RATING_THRESHOLD, RANDOM_SEED, TOP_K_CANDIDATES
from src.data import (
    MOVIELENS_100K_URL,
    calculate_sparsity,
    create_implicit_feedback,
    create_user_item_matrix,
    dataset_overview,
    identify_cold_start_entities,
    interaction_counts,
    load_bundle,
    random_train_test_split,
    time_based_train_test_split,
)

from src.models import baseline_summary, popularity_scores
sns.set_theme(style="whitegrid")
np.random.seed(RANDOM_SEED)

## 1. 데이터 로딩

공식 데이터셋은 `data/raw/ml-100k/`에 위치해야 합니다.  
필요하면 공식 URL(https://files.grouplens.org/datasets/movielens/ml-100k.zip)에서 받아 압축을 풀어둘 수 있습니다.

In [ ]:
# bundle = load_bundle(download_if_missing=True)  # 네트워크가 가능한 환경에서만 사용
bundle = load_bundle(download_if_missing=False)
ratings = bundle.ratings
items = bundle.items
users = bundle.users
genres = bundle.genres

ratings.head()

In [ ]:
overview = dataset_overview(bundle)
pd.Series(overview)

## 2. 평점 분포 분석

In [ ]:
rating_summary = ratings["rating"].describe()
display(rating_summary)

fig, ax = plt.subplots(figsize=(8, 4))
sns.countplot(data=ratings, x="rating", ax=ax, palette="Purples")
ax.set_title("Rating Distribution")
plt.show()

## 3. 유저당 / 아이템당 평점 수 분포

In [ ]:
user_counts, item_counts = interaction_counts(ratings)
display(user_counts.describe().rename("user_rating_count"))
display(item_counts.describe().rename("item_rating_count"))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(user_counts, bins=30, ax=axes[0], color="mediumpurple")
axes[0].set_title("Ratings per User")
axes[0].set_xlabel("# ratings")

sns.histplot(item_counts, bins=30, ax=axes[1], color="slateblue")
axes[1].set_title("Ratings per Item")
axes[1].set_xlabel("# ratings")

plt.tight_layout()
plt.show()

## 4. 희소성(Sparsity)

In [ ]:
sparsity = calculate_sparsity(ratings)
print(f"Sparsity: {sparsity:.4%}")

## 5. 시간에 따른 평점 트렌드

In [ ]:
ratings_by_month = (
    ratings.assign(year_month=ratings["rated_at"].dt.to_period("M").astype(str))
    .groupby("year_month")
    .size()
    .rename("rating_count")
)

fig, ax = plt.subplots(figsize=(12, 4))
ratings_by_month.plot(ax=ax, color="rebeccapurple")
ax.set_title("Ratings Over Time")
ax.set_ylabel("count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 6. Explicit / Implicit 피드백 구성

In [ ]:
explicit_matrix = create_user_item_matrix(ratings, value_column="rating", fill_value=0.0)
implicit_feedback = create_implicit_feedback(ratings, threshold=HIGH_RATING_THRESHOLD)
implicit_matrix = create_user_item_matrix(
    implicit_feedback.rename(columns={"interaction": "implicit_signal"}),
    value_column="implicit_signal",
    fill_value=0.0,
)

print("Explicit matrix shape:", explicit_matrix.shape)
print("Implicit matrix shape:", implicit_matrix.shape)
implicit_feedback.head()

## 7. Cold-start 유저 / 아이템 식별

In [ ]:
cold_start = identify_cold_start_entities(ratings)
print("Cold-start users:", len(cold_start["cold_start_users"]))
print("Cold-start items:", len(cold_start["cold_start_items"]))

## 8. Train / Test 분할 전략 비교

In [ ]:
train_random, test_random = random_train_test_split(ratings, test_size=0.2)
train_time, test_time = time_based_train_test_split(ratings, test_ratio=0.2)

split_summary = pd.DataFrame({
    "split": ["random_train", "random_test", "time_train", "time_test"],
    "rows": [len(train_random), len(test_random), len(train_time), len(test_time)],
    "min_timestamp": [
        train_random["rated_at"].min(),
        test_random["rated_at"].min(),
        train_time["rated_at"].min(),
        test_time["rated_at"].min(),
    ],
    "max_timestamp": [
        train_random["rated_at"].max(),
        test_random["rated_at"].max(),
        train_time["rated_at"].max(),
        test_time["rated_at"].max(),
    ],
})
split_summary

## 9. 베이스라인 준비

In [ ]:
baseline_info = baseline_summary(train_random)
popular_items = popularity_scores(train_random).head(10)
display(pd.Series(baseline_info))
display(popular_items)

## 10. 평가 프레임워크 메모
- 후보 K: `(5, 10, 20)`
- 대표 기본값: `10`
- 최소 추적 지표: RMSE, Precision@K, Recall@K, NDCG@K, MAP
- 추가 지표: Coverage, Intra-list Diversity
- 베이스라인: 전체 평균, 유저 평균, 인기도 기반
- 중요: Top-K 추천 평가는 학습에서 이미 본 아이템을 제외하고 계산

## 다음 단계
1. EDA 결과를 기반으로 베이스라인 추천기를 정의한다.
2. Day 2에서 User-CF / Item-CF / SVD / ALS를 비교한다.
3. 평가 지표 계산은 `src/evaluation/` 유틸을 재사용한다.